**Steps**

1. Load processed radiology reports.
2. Separate development and held-out studies.
3. Tokenise the reports using BioClinicalBERT.
4. Train the text encoder with a temporary classification head.
5. Evaluate the text-only model on the development validation subset.
6. Save the trained model.
7. Extract BioClinicalBERT feature vectors for later multimodal fusion.

### Output

- bioclinicalbert_text_encoder.pt
- text_features.csv
- text_baseline_metrics.csv

In [2]:
#!pip -q install transformers accelerate scikit-learn
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from tqdm.auto import tqdm

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Set paths
base_path = (
    '/content/drive/MyDrive/'
    'dissertation_project/data'
)

processed_path = (
    f'{base_path}/processed'
)

model_path = (
    f'{base_path}/models'
)

os.makedirs(
    model_path,
    exist_ok=True
)

print("Processed path:")
print(processed_path)

print("\nModel path:")
print(model_path)

Processed path:
/content/drive/MyDrive/dissertation_project/data/processed

Model path:
/content/drive/MyDrive/dissertation_project/data/models


In [4]:
# Set random seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


In [5]:
# Check GPU
device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Device: cuda
GPU: Tesla T4


In [6]:
# Load the processed text
text_file = (
    f'{processed_path}/text_processed.csv'
)

text_data = pd.read_csv(
    text_file
)

print(
    "Text dataset:",
    text_data.shape
)

print(
    "Unique studies:",
    text_data['study_id'].nunique()
)

print(
    "\nColumns:"
)

print(
    text_data.columns.tolist()
)

Text dataset: (2200, 10)
Unique studies: 2200

Columns:
['subject_id', 'study_id', 'report_text', 'No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']


In [7]:
# Load the 687 held-out IDs
heldout_ids_file = (
    f'{processed_path}/test_calibration_ids.csv'
)

heldout_ids = pd.read_csv(
    heldout_ids_file
)

heldout_ids['study_id'] = (
    heldout_ids['study_id']
    .astype(str)
)

print(
    "Held-out records:",
    len(heldout_ids)
)

print(
    "Unique held-out studies:",
    heldout_ids['study_id']
    .nunique()
)

Held-out records: 687
Unique held-out studies: 687


In [8]:
# Make sure study IDs have the same type
text_data['study_id'] = (
    text_data['study_id']
    .astype(str)
)

heldout_id_set = set(
    heldout_ids['study_id']
)

In [9]:
# Create the development dataset
development_data = text_data[
    ~text_data['study_id']
    .isin(heldout_id_set)
].copy()

development_data = (
    development_data
    .reset_index(drop=True)
)

print(
    "Development records:",
    len(development_data)
)

print(
    "Held-out records:",
    len(heldout_ids)
)

print(
    "Total:",
    len(development_data)
    + len(heldout_ids)
)

Development records: 1513
Held-out records: 687
Total: 2200


In [10]:
# Define the seven labels
label_columns = [
    'No Finding',
    'Support Devices',
    'Pleural Effusion',
    'Lung Opacity',
    'Atelectasis',
    'Cardiomegaly',
    'Edema'
]

available_labels = [
    label
    for label in label_columns
    if label in development_data.columns
]

print(
    "Labels:"
)

print(
    available_labels
)

Labels:
['No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']


In [11]:
# Remove empty reports
development_data['report_text'] = (
    development_data['report_text']
    .fillna('')
    .astype(str)
)

development_data = development_data[
    development_data['report_text']
    .str.strip()
    .ne('')
].copy()

development_data = (
    development_data
    .reset_index(drop=True)
)

print(
    "Development records with text:",
    len(development_data)
)

Development records with text: 1513


In [12]:
# Convert labels to numeric values
for label in available_labels:

    development_data[label] = pd.to_numeric(
        development_data[label],
        errors='coerce'
    ).fillna(0)

print(
    development_data[
        available_labels
    ].head()
)

   No Finding  Support Devices  Pleural Effusion  Lung Opacity  Atelectasis  \
0         1.0              0.0               0.0           0.0          0.0   
1         1.0              0.0               0.0           0.0          0.0   
2         1.0              0.0               0.0           0.0          0.0   
3         0.0              1.0               0.0           0.0          0.0   
4         1.0              0.0               0.0           0.0          0.0   

   Cardiomegaly  Edema  
0           0.0    0.0  
1           0.0    0.0  
2           0.0    0.0  
3           0.0    0.0  
4           0.0    0.0  


In [13]:
# Internal train/validation split
train_data, validation_data = train_test_split(
    development_data,
    test_size=0.15,
    random_state=SEED
)

train_data = (
    train_data
    .reset_index(drop=True)
)

validation_data = (
    validation_data
    .reset_index(drop=True)
)

print(
    "Training records:",
    len(train_data)
)

print(
    "Validation records:",
    len(validation_data)
)

Training records: 1286
Validation records: 227


In [14]:
# Load BioClinicalBERT tokenizer
MODEL_NAME = (
    'emilyalsentzer/'
    'Bio_ClinicalBERT'
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(
    "Tokenizer loaded:"
)

print(
    MODEL_NAME
)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Tokenizer loaded:
emilyalsentzer/Bio_ClinicalBERT


In [15]:
# Create dataset class
class ClinicalTextDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        labels,
        max_length=256
    ):

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.tokenizer = tokenizer
        self.labels = labels
        self.max_length = max_length

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        text = str(
            row['report_text']
        )

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {
            'input_ids':
                encoding['input_ids'].squeeze(0),

            'attention_mask':
                encoding['attention_mask'].squeeze(0)
        }

        labels = np.array(
            [
                row[label]
                for label in self.labels
            ],
            dtype=np.float32
        )

        item['labels'] = torch.tensor(
            labels,
            dtype=torch.float32
        )

        return item

In [16]:
# Create datasets
MAX_LENGTH = 256

train_dataset = ClinicalTextDataset(
    train_data,
    tokenizer,
    available_labels,
    MAX_LENGTH
)

validation_dataset = ClinicalTextDataset(
    validation_data,
    tokenizer,
    available_labels,
    MAX_LENGTH
)

print(
    "Training dataset:",
    len(train_dataset)
)

print(
    "Validation dataset:",
    len(validation_dataset)
)

Training dataset: 1286
Validation dataset: 227


In [17]:
# Create DataLoaders
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

Training batches: 161
Validation batches: 29


In [18]:
# Define BioClinicalBERT classifier
class BioClinicalBERTClassifier(
    nn.Module
):

    def __init__(
        self,
        model_name,
        num_labels
    ):

        super().__init__()

        self.bert = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = (
            self.bert.config.hidden_size
        )

        self.dropout = nn.Dropout(
            0.2
        )

        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # CLS representation
        cls_embedding = (
            outputs.last_hidden_state[:, 0, :]
        )

        cls_embedding = self.dropout(
            cls_embedding
        )

        logits = self.classifier(
            cls_embedding
        )

        return logits, cls_embedding

In [19]:
# Create model
model = BioClinicalBERTClassifier(
    MODEL_NAME,
    len(available_labels)
)

model = model.to(device)

print(
    "Model loaded."
)

print(
    "Number of labels:",
    len(available_labels)
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Number of labels: 7


In [20]:
# Calculate class weights
y_train = (
    train_data[
        available_labels
    ]
    .values
    .astype(np.float32)
)

positive_counts = (
    y_train.sum(axis=0)
)

negative_counts = (
    len(y_train)
    - positive_counts
)

pos_weights = (
    negative_counts
    / np.maximum(
        positive_counts,
        1
    )
)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32
).to(device)

print(
    pd.DataFrame({
        'label': available_labels,
        'positive': positive_counts,
        'negative': negative_counts,
        'weight': pos_weights.cpu().numpy()
    })
)

              label  positive  negative     weight
0        No Finding     347.0     939.0   2.706052
1   Support Devices     484.0     802.0   1.657025
2  Pleural Effusion     334.0     952.0   2.850299
3      Lung Opacity     330.0     956.0   2.896970
4       Atelectasis     236.0    1050.0   4.449152
5      Cardiomegaly     302.0     984.0   3.258278
6             Edema     106.0    1180.0  11.132075


In [21]:
# Loss and optimizer
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

print("Optimizer ready.")

Optimizer ready.


In [22]:
# Training function
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0

    progress = tqdm(
        loader,
        desc='Training'
    )

    for batch in progress:

        input_ids = (
            batch['input_ids']
            .to(device)
        )

        attention_mask = (
            batch['attention_mask']
            .to(device)
        )

        labels = (
            batch['labels']
            .to(device)
        )

        optimizer.zero_grad()

        logits, _ = model(
            input_ids,
            attention_mask
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item()
            * input_ids.size(0)
        )

        progress.set_postfix(
            loss=loss.item()
        )

    return (
        total_loss
        / len(loader.dataset)
    )

In [23]:
# Validation function
def evaluate_model(
    model,
    loader,
    device
):

    model.eval()

    all_labels = []
    all_probabilities = []

    total_loss = 0

    with torch.no_grad():

        for batch in loader:

            input_ids = (
                batch['input_ids']
                .to(device)
            )

            attention_mask = (
                batch['attention_mask']
                .to(device)
            )

            labels = (
                batch['labels']
                .to(device)
            )

            logits, _ = model(
                input_ids,
                attention_mask
            )

            probabilities = torch.sigmoid(
                logits
            )

            loss = criterion(
                logits,
                labels
            )

            total_loss += (
                loss.item()
                * input_ids.size(0)
            )

            all_labels.append(
                labels.cpu().numpy()
            )

            all_probabilities.append(
                probabilities.cpu().numpy()
            )

    y_true = np.vstack(
        all_labels
    )

    y_prob = np.vstack(
        all_probabilities
    )

    y_pred = (
        y_prob >= 0.5
    ).astype(int)

    return (
        total_loss / len(loader.dataset),
        y_true,
        y_prob,
        y_pred
    )

In [24]:
# Train
EPOCHS = 3

history = []

for epoch in range(EPOCHS):

    print(
        f"\n========== Epoch "
        f"{epoch + 1}/{EPOCHS} =========="
    )

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, y_true, y_prob, y_pred = (
        evaluate_model(
            model,
            validation_loader,
            device
        )
    )

    print(
        f"Training loss: {train_loss:.4f}"
    )

    print(
        f"Validation loss: {val_loss:.4f}"
    )

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'validation_loss': val_loss
    })


========== Epoch 1/3 ==========


Training:   0%|          | 0/161 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Training loss: 0.8849
Validation loss: 0.5055

========== Epoch 2/3 ==========


Training:   0%|          | 0/161 [00:00<?, ?it/s]

Training loss: 0.3431
Validation loss: -0.1114

========== Epoch 3/3 ==========


Training:   0%|          | 0/161 [00:00<?, ?it/s]

Training loss: -0.3465
Validation loss: -0.4647


In [25]:
# Display training history
history_df = pd.DataFrame(
    history
)

display(
    history_df
)

,epoch,train_loss,validation_loss
0,1,0.884923,0.505471
1,2,0.343067,-0.111362
2,3,-0.346515,-0.464686


In [26]:
# Calculate baseline metrics
metrics = []

for i, label in enumerate(available_labels):

    # Convert ground truth to binary
    true_values = (
        y_true[:, i] == 1
    ).astype(int)

    # Model predictions are already probabilities
    probabilities = y_prob[:, i]

    # Convert probabilities to binary predictions
    predicted_values = (
        probabilities >= 0.5
    ).astype(int)

    accuracy = accuracy_score(
        true_values,
        predicted_values
    )

    precision = precision_score(
        true_values,
        predicted_values,
        average='binary',
        zero_division=0
    )

    recall = recall_score(
        true_values,
        predicted_values,
        average='binary',
        zero_division=0
    )

    f1 = f1_score(
        true_values,
        predicted_values,
        average='binary',
        zero_division=0
    )

    try:

        roc_auc = roc_auc_score(
            true_values,
            probabilities
        )

    except ValueError:

        roc_auc = np.nan

    metrics.append({
        'label': label,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc
    })


baseline_metrics = pd.DataFrame(
    metrics
)

display(baseline_metrics)

,label,accuracy,precision,recall,f1,roc_auc
0,No Finding,0.823789,0.606742,0.915254,0.729730,0.946025
1,Support Devices,0.806167,0.660000,0.868421,0.750000,0.867811
2,Pleural Effusion,0.757709,0.507937,0.571429,0.537815,0.829470
3,Lung Opacity,0.889868,0.783784,0.865672,0.822695,0.952146
4,Atelectasis,0.845815,0.754386,0.671875,0.710744,0.927818
5,Cardiomegaly,0.762115,0.505747,0.800000,0.619718,0.908774
6,Edema,0.920705,0.540541,0.952381,0.689655,0.957466


In [27]:
# Overall binary metrics

y_true_binary = (
    y_true == 1
).astype(int)

y_pred_binary = (
    y_prob >= 0.5
).astype(int)


overall_metrics = pd.DataFrame({
    'metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'ROC-AUC'
    ],

    'value': [
        accuracy_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten()
        ),

        precision_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            average='binary',
            zero_division=0
        ),

        recall_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            average='binary',
            zero_division=0
        ),

        f1_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            average='binary',
            zero_division=0
        ),

        roc_auc_score(
            y_true_binary,
            y_prob,
            average='macro'
        )
    ]
})

display(
    overall_metrics
)

,metric,value
0,Accuracy,0.829452
1,Precision,0.625247
2,Recall,0.796482
3,F1,0.700552
4,ROC-AUC,0.912787


In [28]:
# Save
metrics_file = (
    f'{processed_path}/'
    'text_baseline_metrics.csv'
)

baseline_metrics.to_csv(
    metrics_file,
    index=False
)

print(
    "Saved:",
    metrics_file
)

history_file = (
    f'{processed_path}/'
    'text_training_history.csv'
)

history_df.to_csv(
    history_file,
    index=False
)

print(
    "Saved:",
    history_file
)


model_file = (
    f'{model_path}/'
    'bioclinicalbert_text_encoder.pt'
)

torch.save(
    {
        'model_state_dict':
            model.state_dict(),

        'model_name':
            MODEL_NAME,

        'labels':
            available_labels,

        'max_length':
            MAX_LENGTH
    },
    model_file
)

print(
    "Model saved:"
)

print(
    model_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/text_baseline_metrics.csv
Saved: /content/drive/MyDrive/dissertation_project/data/processed/text_training_history.csv
Model saved:
/content/drive/MyDrive/dissertation_project/data/models/bioclinicalbert_text_encoder.pt


In [29]:
# Extract BioClinicalBERT features
def extract_text_features(
    model,
    dataframe,
    tokenizer,
    device,
    max_length=256,
    batch_size=8
):

    dataset = ClinicalTextDataset(
        dataframe,
        tokenizer,
        available_labels,
        max_length
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model.eval()

    feature_list = []

    with torch.no_grad():

        for batch in tqdm(
            loader,
            desc='Extracting text features'
        ):

            input_ids = (
                batch['input_ids']
                .to(device)
            )

            attention_mask = (
                batch['attention_mask']
                .to(device)
            )

            _, features = model(
                input_ids,
                attention_mask
            )

            feature_list.append(
                features.cpu().numpy()
            )

    return np.vstack(
        feature_list
    )

In [30]:
# Extract features
train_features = extract_text_features(
    model,
    development_data,
    tokenizer,
    device,
    MAX_LENGTH,
    BATCH_SIZE
)

print(
    "Development feature shape:",
    train_features.shape
)

Extracting text features:   0%|          | 0/190 [00:00<?, ?it/s]

Development feature shape: (1513, 768)


In [31]:
# Extract features for the 687 held-out records
heldout_text = text_data[
    text_data['study_id']
    .isin(heldout_id_set)
].copy()

heldout_text = (
    heldout_text
    .drop_duplicates(
        subset='study_id'
    )
    .reset_index(drop=True)
)

print(
    "Held-out text records:",
    len(heldout_text)
)

heldout_features = extract_text_features(
    model,
    heldout_text,
    tokenizer,
    device,
    MAX_LENGTH,
    BATCH_SIZE
)

print(
    "Held-out feature shape:",
    heldout_features.shape
)

Held-out text records: 687


Extracting text features:   0%|          | 0/86 [00:00<?, ?it/s]

Held-out feature shape: (687, 768)


In [32]:
# Save development features
development_feature_df = pd.DataFrame(
    train_features,
    columns=[
        f'text_feature_{i}'
        for i in range(
            train_features.shape[1]
        )
    ]
)

development_feature_df.insert(
    0,
    'study_id',
    development_data[
        'study_id'
    ].values
)

development_feature_file = (
    f'{processed_path}/'
    'text_features_development.csv'
)

development_feature_df.to_csv(
    development_feature_file,
    index=False
)

print(
    "Saved:",
    development_feature_file
)

heldout_feature_df = pd.DataFrame(
    heldout_features,
    columns=[
        f'text_feature_{i}'
        for i in range(
            heldout_features.shape[1]
        )
    ]
)

heldout_feature_df.insert(
    0,
    'study_id',
    heldout_text[
        'study_id'
    ].values
)

heldout_feature_file = (
    f'{processed_path}/'
    'text_features_heldout.csv'
)

heldout_feature_df.to_csv(
    heldout_feature_file,
    index=False
)

print(
    "Saved:",
    heldout_feature_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/text_features_development.csv
Saved: /content/drive/MyDrive/dissertation_project/data/processed/text_features_heldout.csv
